# YOLOv8 Surface Defect Detection Training
MVTec AD Dataset | Vision Inspection Portfolio

In [ ]:
!pip install ultralytics
!pip install torch torchvision

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Upload Dataset
Upload your data/processed/bottle/ folder to Google Drive
Expected path: /content/drive/MyDrive/vision_portfolio/bottle/

In [ ]:
import os
from pathlib import Path

data_dir = Path('/content/drive/MyDrive/vision_portfolio/bottle')
yaml_path = data_dir / 'dataset.yaml'

print(f"Dataset path exists: {data_dir.exists()}")
print(f"YAML exists: {yaml_path.exists()}")

train_images = list((data_dir / 'images/train').glob('*.png'))
val_images = list((data_dir / 'images/val').glob('*.png'))
print(f"Train images: {len(train_images)}")
print(f"Val images: {len(val_images)}")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=640,
    batch=16,
    name='bottle_test',
    project='/content/drive/MyDrive/vision_portfolio/runs',
    patience=5,
    save=True,
    plots=True
)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/vision_portfolio/runs/bottle_test')

metrics = ['results.png', 'confusion_matrix.png', 'PR_curve.png']
for m in metrics:
    img_path = run_dir / m
    if img_path.exists():
        img = plt.imread(img_path)
        plt.figure(figsize=(12,8))
        plt.imshow(img)
        plt.title(m)
        plt.axis('off')
        plt.show()

print(f"Best mAP50: {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")

In [ ]:
best_model = YOLO('/content/drive/MyDrive/vision_portfolio/runs/bottle_test/weights/best.pt')
best_model.export(format='onnx', imgsz=640, simplify=True)
print("ONNX export complete")